## Import Libraries and Load Data

In [1]:
!pip install -q transformers datasets evaluate accelerate sentencepiece

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate
import torch

In [3]:
train_pairs = pd.read_csv("train_pairs.csv")

## Train Validation Split

In [4]:
train_df, valid_df = train_test_split(
    train_pairs,
    test_size=0.2,
    random_state=42,
    stratify=train_pairs["label"]
)

## Load Tokenizer

In [10]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## Create HuggingFace Dataset

In [11]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

In [12]:
train_dataset[0]

{'id': 1849,
 'prompt': 'What is the Josephson effect?',
 'option': 'The Josephson effect is a phenomenon exploited by magnetic devices such as SQUIDs. It is used in the most accurate available measurements of the electric flux quantum Φ0 = h/(2e), where h is the magnetic constant.',
 'option_id': 'E',
 'label': 0,
 '__index_level_0__': 9244}

## Tokenizer

In [13]:
def tokenize(example):
    return tokenizer(
        example["prompt"],
        example["option"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [14]:
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [15]:
train_dataset = train_dataset.rename_column("label", "labels")
valid_dataset = valid_dataset.rename_column("label", "labels")

In [16]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

valid_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

## Load The Model

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Metric

In [18]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

## Training Arguments

In [20]:
training_args = TrainingArguments(
    output_dir="./bert-base-uncased",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=100,

    report_to="none"
)

## Trainer

In [21]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics
)

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.425613,0.387164,0.845500,0.420263
2,0.228672,0.179171,0.930000,0.806094
3,0.147919,0.141184,0.947000,0.856757


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1500, training_loss=0.3057987791697184, metrics={'train_runtime': 1252.9965, 'train_samples_per_second': 19.154, 'train_steps_per_second': 1.197, 'total_flos': 3157332664320000.0, 'train_loss': 0.3057987791697184, 'epoch': 3.0})

In [23]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.147919,0.141184,3,0.947000,0.856757


{'eval_loss': 0.14118388295173645, 'eval_accuracy': 0.947, 'eval_f1': 0.8567567567567568}


In [25]:
trainer.save_model("bert_model")

tokenizer.save_pretrained("bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_model/tokenizer_config.json', 'bert_model/tokenizer.json')